# 04 — Review QC and record explicit decisions

**You will:** inspect flagged observations in context, distinguish a proposal from a
release decision, and save reviewed QC that FixDataset can consume.
This notebook deliberately requires your decisions. It does not label every detector
flag as a bad observation. The later apply notebook refuses an unfinished review.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, tarfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tomlkit
COURSE = Path(os.environ['PLEB_COURSE_HOME'])
sys.path.insert(0, str(COURSE))
import release_tools as lab
ROOT = lab.environment(Path(os.environ['PLEB_LEARNING_OUTPUT']))
REPO = ROOT/'dataset-repo'
CONFIGS = ROOT/'configs'
PULSARS = lab.PULSARS
CLI = [sys.executable, '-m', 'pleb.cli']

In [ ]:
from pleb import qc_review
frames = []
paths = json.loads((ROOT/'qc_paths.json').read_text())
for psr, path in paths.items():
    # Relative table paths are part of PLEB's review ID. Keep this input root
    # and the original CSVs frozen when transferring decisions.
    frame = qc_review.load_qc_csv(path, root=ROOT/'results/03_detect')
    frame['training_pulsar'] = psr
    frames.append(frame)
table = pd.concat(frames,ignore_index=True)
assert table.review_id.is_unique
(ROOT/'qc_input_hashes.json').write_text(json.dumps({
    str(Path(p).relative_to(ROOT/'results/03_detect')):hashlib.sha256(Path(p).read_bytes()).hexdigest()
    for p in paths.values()},indent=2))
evidence = [c for c in ['robust_outlier','robust_global_outlier','bad_point','bad_mad'] if c in table]
assert evidence, 'Inspect QC schema before defining a review proposal'
proposed = table[evidence].fillna(False).astype(bool).any(axis=1)
candidates = table.loc[proposed].copy()
columns = [c for c in ['review_id','mjd','freq','sys','robust_z','robust_global_z']+evidence if c in table]
display(candidates[columns])
print('QC columns available for plotting:',list(table.columns))

Inspect time, frequency, backend and uncertainty together. A large residual with a
large error bar has a different meaning from the same residual with a small error bar.
Do not hide a coherent event by centring each backend or narrowing a plot without a full-range view.
The helper below exports fresh matched Tempo2 residuals on the prepared snapshot, then
overlays the proposed candidates by MJD. This plot is evidence for your review, not an extra detector.

In [ ]:
for psr in PULSARS:
    data = ROOT/'exports/02_prepared/data'/psr
    work = lab.simulation.project(ROOT/'review_fits'/psr)
    residuals, stats = lab.simulation.fit_tempo2(work,data/f'{psr}.par',data/f'{psr}_all.tim',
                                               'review_baseline',epoch='58730')
    fig = lab.simulation.plot_residuals(residuals,psr+': prepared baseline')
    for mjd in candidates.loc[candidates.training_pulsar==psr,'mjd']:
        fig.axes[0].axvline(mjd,color='red',alpha=.3)
    display(pd.DataFrame([stats]))
display(candidates[columns])

## Enter your decisions

Copy the `review_id` values into the lists below. For each proposed candidate choose
`mark_bad` or `keep`; explain why. Keeping a candidate is an explicit override. A real
review must also inspect unflagged data and may identify additional bad observations.
IDs come from `pleb.qc_review`, including the relative source-table path, row index
and measurement fields. They identify this frozen review input; they are not permanent
IDs across regenerated or reordered QC tables. Keep the input hashes and original columns.
The final residual-review lesson also captures physical TIM paths, line numbers and row hashes.

`make_override_rows` records action, reason, reviewer and UTC timestamp. `append_overrides`
keeps history; a later decision for the same review ID takes effect without erasing the
earlier one. Unreviewed rows must not be labelled as explicit human keeps.

Change `REVIEW_COMPLETE` only after inspecting the plots and table. This flag records
your checkpoint; it is not a statistical certificate. Edit the reason/reviewer fields
for real work. No row is deleted and no decision is packed into dozens of TIM flags.

In [ ]:
MARK_BAD = []       # copy the PLEB review_id values displayed above
KEEP_CANDIDATES = []
REVIEWER = ''
REASON = ''
REVIEW_COMPLETE = False

assert REVIEW_COMPLETE and REVIEWER and REASON, 'Complete your review before saving approved decisions'
bad, keep = set(MARK_BAD), set(KEEP_CANDIDATES)
assert not bad & keep
assert bad | keep <= set(table.review_id)
assert set(candidates.review_id) <= bad | keep, 'Some proposed candidates are unresolved'
override_path = ROOT/'review/overrides.csv'
history = qc_review.load_overrides(override_path)
for action, ids in [('mark_bad',bad),('keep',keep)]:
    decisions = qc_review.make_override_rows(table.loc[table.review_id.isin(ids)],
        action=action, reason=REASON, reviewer=REVIEWER, source='release_training_notebook')
    history = qc_review.append_overrides(history,decisions)
qc_review.write_overrides(history,override_path)
reviewed = qc_review.apply_overrides(table,history)
assert (reviewed.manual_action.ne('') == reviewed.review_id.isin(bad|keep)).all()
for psr in PULSARS:
    target = ROOT/'review'/f'{psr}_qc.csv'
    reviewed.loc[reviewed.training_pulsar==psr].to_csv(target,index=False)
approval = dict(training_only=True,reviewer=REVIEWER,reason=REASON,
    selected_bad=sorted(bad),explicit_keeps=sorted(keep),files=lab.hashes(ROOT/'review'),
    qc_input_hashes=json.loads((ROOT/'qc_input_hashes.json').read_text()))
(ROOT/'review_approval.json').write_text(json.dumps(approval,indent=2))
display(reviewed.groupby(['training_pulsar','manual_action']).size())

## Exercise: change a decision without erasing history

This detached example uses a copy of one row. It does not change your saved review.
An initial rejection is followed by an explicit keep. The automatic evidence remains
visible, both decisions remain in the history, and the most recent action wins.

In [ ]:
example = table.iloc[[0]].copy()
example['bad_point'] = True  # artificial detector proposal for this example only
example['auto_decision'] = 'BAD_TOA'
rejected = qc_review.make_override_rows(example,action='mark_bad',
    reason='Training example: initial rejection',reviewer='Training reviewer',
    timestamp='2026-01-01T00:00:00+00:00',source='detached_training_example')
accepted = qc_review.make_override_rows(example,action='keep',
    reason='Training example: inspected and accepted',reviewer='Training reviewer',
    timestamp='2026-01-01T00:01:00+00:00',source='detached_training_example')
demo_history = qc_review.append_overrides(rejected,accepted)
demo_review = qc_review.apply_overrides(example,demo_history)
assert len(demo_history)==2 and demo_review.bad_point.all()
assert demo_review.auto_decision.eq('BAD_TOA').all()
assert demo_review.reviewed_keep.all() and not demo_review.reviewed_bad_point.any()
display(demo_history[['manual_action','manual_reason','reviewer','reviewed_at']])

**Your task:** explain why a reviewed keep must survive a large automatic score.
After choosing decisions, you may inspect `raw_manifest.json` to compare with the injected
truth. That truth is a teaching check; it is not available to reviewers of real observations.